Notebook script to analyze model checkpoints and json files

In [1]:
import numpy as np
import json
import sys
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sem_proj.data.datasets import BoasSequenceDataset
from sem_proj.data.preprocessing import PreprocessingConfig
from sem_proj.models.model_factory import SSLEpochTransformerConv1D_v2, SequenceGRUClassifier
from sem_proj.data.boa_loader import build_pid_mappings

CHECKPOINT_LEOMED_DIR = PROJECT_ROOT / "checkpoints_leomed"
JSON_DIR = PROJECT_ROOT / "reports" / "metrics"
TARGET_DIR = PROJECT_ROOT / "plots"

In [ ]:
### check if stronger MLP v1 results are better than with the previous MLP
ctxfree_finetuned_stronger_MLP_v1_json = JSON_DIR / "ctxfree_finetuning_results_val_step_stronger_MLP_v1.json"
ctxfree_fullysupervised_stronger_MLP_v1_json = JSON_DIR / "ctxfree_fullysupervised_results_val_step_stronger_MLP_v1.json"
with open(ctxfree_finetuned_stronger_MLP_v1_json, 'r') as f:
    ctxfree_finetuned_stronger_MLP_v1 = json.load(f)
with open(ctxfree_fullysupervised_stronger_MLP_v1_json, 'r') as f:
    ctxfree_fullysupervised_stronger_MLP_v1 = json.load(f)
mf1_finetuned = []
mf1_fullysuperv = []

for key in ctxfree_finetuned_stronger_MLP_v1:
    nested_dict = ctxfree_finetuned_stronger_MLP_v1[key]
    mf1_score = nested_dict['stage1_mf1']
    mf1_finetuned.append(mf1_score)
for key in ctxfree_fullysupervised_stronger_MLP_v1:
    nested_dict = ctxfree_fullysupervised_stronger_MLP_v1[key]
    mf1_score = nested_dict['stage1_mf1']
    mf1_fullysuperv.append(mf1_score)

p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage

plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxfree_finetuning_vs_fullysupervised_varying_p_stronger_MLP_v1.pdf', dpi=300, bbox_inches='tight')


Now context-sensitive model, uni- and bidirectional GRU, 'new' MF1

L=20, s=5 during training

In [4]:
# bidirectional L20 s5
ctxsensitive_finetuned_bidir_json = JSON_DIR / "ctxsensitive_finetuning_results_bidirTrue_L20_s5_val_step_new.json"
ctxsensitive_fullysupervised_bidir_json = JSON_DIR / "ctxsensitive_fullysupervised_results_bidirTrue_L20_s5_val_step_new.json"
with open(ctxsensitive_finetuned_bidir_json, 'r') as f:
    ctxsensitive_finetuned_bidir_l20_s5 = json.load(f)
with open(ctxsensitive_fullysupervised_bidir_json, 'r') as f:
    ctxsensitive_fullysupervised_bidir_l20_s5 = json.load(f)
mf1_finetuned_bidir_l20_s5 = []
mf1_fullysuperv_bidir_l20_s5 = []

for key in ctxsensitive_finetuned_bidir_l20_s5:
    nested_dict = ctxsensitive_finetuned_bidir_l20_s5[key]
    mf1_score = nested_dict['val_mf1']
    mf1_finetuned_bidir_l20_s5.append(mf1_score)
for key in ctxsensitive_fullysupervised_bidir_l20_s5:
    nested_dict = ctxsensitive_fullysupervised_bidir_l20_s5[key]
    mf1_score = nested_dict['val_mf1']
    mf1_fullysuperv_bidir_l20_s5.append(mf1_score)
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage

plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_bidir_l20_s5, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv_bidir_l20_s5, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_finetuning_vs_fullysupervised_val_step_varying_p_bidir_l20_s5.pdf', dpi=300, bbox_inches='tight')


# unidirectional L20 s5
ctxsensitive_finetuned_unidir_json = JSON_DIR / "ctxsensitive_finetuning_results_bidirFalse_L20_s5_val_step_new.json"
ctxsensitive_fullysupervised_unidir_json = JSON_DIR / "ctxsensitive_fullysupervised_results_bidirFalse_L20_s5_val_step_new.json"
with open(ctxsensitive_finetuned_unidir_json, 'r') as f:
    ctxsensitive_finetuned_unidir_l20_s5 = json.load(f)
with open(ctxsensitive_fullysupervised_unidir_json, 'r') as f:
    ctxsensitive_fullysupervised_unidir_l20_s5 = json.load(f)
mf1_finetuned_unidir_l20_s5 = []
mf1_fullysuperv_unidir_l20_s5 = []

for key in ctxsensitive_finetuned_unidir_l20_s5:
    nested_dict = ctxsensitive_finetuned_unidir_l20_s5[key]
    mf1_score = nested_dict['val_mf1']
    mf1_finetuned_unidir_l20_s5.append(mf1_score)
for key in ctxsensitive_fullysupervised_unidir_l20_s5:
    nested_dict = ctxsensitive_fullysupervised_unidir_l20_s5[key]
    mf1_score = nested_dict['val_mf1']
    mf1_fullysuperv_unidir_l20_s5.append(mf1_score)
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage

plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_unidir_l20_s5, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv_unidir_l20_s5, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_finetuning_vs_fullysupervised_val_step_varying_p_unidir_l20_s5.pdf', dpi=300, bbox_inches='tight')

L=10, s=2 during training

In [5]:
# bidirectional L10 s2
ctxsensitive_finetuned_bidir_json = JSON_DIR / "ctxsensitive_finetuning_results_bidirTrue_L10_s2_val_step_new.json"
ctxsensitive_fullysupervised_bidir_json = JSON_DIR / "ctxsensitive_fullysupervised_results_bidirTrue_L10_s2_val_step_new.json"
with open(ctxsensitive_finetuned_bidir_json, 'r') as f:
    ctxsensitive_finetuned_bidir_l10_s2 = json.load(f)
with open(ctxsensitive_fullysupervised_bidir_json, 'r') as f:
    ctxsensitive_fullysupervised_bidir_l10_s2 = json.load(f)
mf1_finetuned_bidir_l10_s2 = []
mf1_fullysuperv_bidir_l10_s2 = []

for key in ctxsensitive_finetuned_bidir_l10_s2:
    nested_dict = ctxsensitive_finetuned_bidir_l10_s2[key]
    mf1_score = nested_dict['val_mf1']
    mf1_finetuned_bidir_l10_s2.append(mf1_score)
for key in ctxsensitive_fullysupervised_bidir_l10_s2:
    nested_dict = ctxsensitive_fullysupervised_bidir_l10_s2[key]
    mf1_score = nested_dict['val_mf1']
    mf1_fullysuperv_bidir_l10_s2.append(mf1_score)
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage

plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_bidir_l10_s2, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv_bidir_l10_s2, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_finetuning_vs_fullysupervised_val_step_varying_p_bidir_l10_s2.pdf', dpi=300, bbox_inches='tight')


# unidirectional L10 s2
ctxsensitive_finetuned_unidir_json = JSON_DIR / "ctxsensitive_finetuning_results_bidirFalse_L10_s2_val_step_new.json"
ctxsensitive_fullysupervised_unidir_json = JSON_DIR / "ctxsensitive_fullysupervised_results_bidirFalse_L10_s2_val_step_new.json"
with open(ctxsensitive_finetuned_unidir_json, 'r') as f:
    ctxsensitive_finetuned_unidir_l10_s2 = json.load(f)
with open(ctxsensitive_fullysupervised_unidir_json, 'r') as f:
    ctxsensitive_fullysupervised_unidir_l10_s2 = json.load(f)
mf1_finetuned_unidir_l10_s2 = []
mf1_fullysuperv_unidir_l10_s2 = []

for key in ctxsensitive_finetuned_unidir_l10_s2:
    nested_dict = ctxsensitive_finetuned_unidir_l10_s2[key]
    mf1_score = nested_dict['val_mf1']
    mf1_finetuned_unidir_l10_s2.append(mf1_score)
for key in ctxsensitive_fullysupervised_unidir_l10_s2:
    nested_dict = ctxsensitive_fullysupervised_unidir_l10_s2[key]
    mf1_score = nested_dict['val_mf1']
    mf1_fullysuperv_unidir_l10_s2.append(mf1_score)
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage

plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_unidir_l10_s2, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv_unidir_l10_s2, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_finetuning_vs_fullysupervised_val_step_varying_p_unidir_l10_s2.pdf', dpi=300, bbox_inches='tight')

L=5, s=1

In [6]:
# bidirectional L5 s1
ctxsensitive_finetuned_bidir_json = JSON_DIR / "ctxsensitive_finetuning_results_bidirTrue_L5_s1_val_step_new.json"
ctxsensitive_fullysupervised_bidir_json = JSON_DIR / "ctxsensitive_fullysupervised_results_bidirTrue_L5_s1_val_step_new.json"
with open(ctxsensitive_finetuned_bidir_json, 'r') as f:
    ctxsensitive_finetuned_bidir_l5_s1 = json.load(f)
with open(ctxsensitive_fullysupervised_bidir_json, 'r') as f:
    ctxsensitive_fullysupervised_bidir_l5_s1 = json.load(f)
mf1_finetuned_bidir_l5_s1 = []
mf1_fullysuperv_bidir_l5_s1 = []

for key in ctxsensitive_finetuned_bidir_l5_s1:
    nested_dict = ctxsensitive_finetuned_bidir_l5_s1[key]
    mf1_score = nested_dict['val_mf1']
    mf1_finetuned_bidir_l5_s1.append(mf1_score)
for key in ctxsensitive_fullysupervised_bidir_l5_s1:
    nested_dict = ctxsensitive_fullysupervised_bidir_l5_s1[key]
    mf1_score = nested_dict['val_mf1']
    mf1_fullysuperv_bidir_l5_s1.append(mf1_score)
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage

plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_bidir_l5_s1, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv_bidir_l5_s1, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_finetuning_vs_fullysupervised_val_step_varying_p_bidir_l5_s1.pdf', dpi=300, bbox_inches='tight')


# unidirectional L5 s1
ctxsensitive_finetuned_unidir_json = JSON_DIR / "ctxsensitive_finetuning_results_bidirFalse_L5_s1_val_step_new.json"
ctxsensitive_fullysupervised_unidir_json = JSON_DIR / "ctxsensitive_fullysupervised_results_bidirFalse_L5_s1_val_step_new.json"
with open(ctxsensitive_finetuned_unidir_json, 'r') as f:
    ctxsensitive_finetuned_unidir_l5_s1 = json.load(f)
with open(ctxsensitive_fullysupervised_unidir_json, 'r') as f:
    ctxsensitive_fullysupervised_unidir_l5_s1 = json.load(f)
mf1_finetuned_unidir_l5_s1 = []
mf1_fullysuperv_unidir_l5_s1 = []

for key in ctxsensitive_finetuned_unidir_l5_s1:
    nested_dict = ctxsensitive_finetuned_unidir_l5_s1[key]
    mf1_score = nested_dict['val_mf1']
    mf1_finetuned_unidir_l5_s1.append(mf1_score)
for key in ctxsensitive_fullysupervised_unidir_l5_s1:
    nested_dict = ctxsensitive_fullysupervised_unidir_l5_s1[key]
    mf1_score = nested_dict['val_mf1']
    mf1_fullysuperv_unidir_l5_s1.append(mf1_score)
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage

plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_unidir_l5_s1, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv_unidir_l5_s1, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_finetuning_vs_fullysupervised_val_step_varying_p_unidir_l5_s1.pdf', dpi=300, bbox_inches='tight')

group the results into one or two (uni and bidir seperately) plots

In [10]:
# bidir L20 s5, L10 s2, L5 s1 combined plot
plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_bidir_l20_s5, marker='x', linestyle='-', linewidth=2, markersize=8, label='SSL (L=20, s=5)', color='green')
plt.plot(p, mf1_fullysuperv_bidir_l20_s5, marker='o',linestyle='-', linewidth=2, markersize=8, label='Supervised (L=20, s=5)', color='red')
plt.plot(p, mf1_finetuned_bidir_l10_s2, marker='x', linestyle='--', linewidth=2, markersize=8, label='SSL (L=10, s=2)', color='green')
plt.plot(p, mf1_fullysuperv_bidir_l10_s2, marker='o', linestyle='--', linewidth=2, markersize=8, label='Supervised (L=10, s=2)', color='red')
plt.plot(p, mf1_finetuned_bidir_l5_s1, marker='x', linestyle=':', linewidth=2, markersize=8, label='SSL (L=5, s=1)', color='green')
plt.plot(p, mf1_fullysuperv_bidir_l5_s1, marker='o', linestyle=':', linewidth=2, markersize=8, label='Supervised (L=5, s=1)', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_finetuning_vs_fullysupervised_val_step_varying_p_bidir_all_configs.pdf', dpi=300, bbox_inches='tight')

In [11]:
# unidir L20 s5, L10 s2, L5 s1 combined plot
plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_unidir_l20_s5, marker='x', linestyle='-', linewidth=2, markersize=8, label='SSL (L=20, s=5)', color='green')
plt.plot(p, mf1_fullysuperv_unidir_l20_s5, marker='o',linestyle='-', linewidth=2, markersize=8, label='Supervised (L=20, s=5)', color='red')
plt.plot(p, mf1_finetuned_unidir_l10_s2, marker='x', linestyle='--', linewidth=2, markersize=8, label='SSL (L=10, s=2)', color='green')
plt.plot(p, mf1_fullysuperv_unidir_l10_s2, marker='o', linestyle='--', linewidth=2, markersize=8, label='Supervised (L=10, s=2)', color='red')
plt.plot(p, mf1_finetuned_unidir_l5_s1, marker='x', linestyle=':', linewidth=2, markersize=8, label='SSL (L=5, s=1)', color='green')
plt.plot(p, mf1_fullysuperv_unidir_l5_s1, marker='o', linestyle=':', linewidth=2, markersize=8, label='Supervised (L=5, s=1)', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_finetuning_vs_fullysupervised_val_step_varying_p_unidir_all_configs.pdf', dpi=300, bbox_inches='tight')